# 🦜🕸️ Agentic RAG with LangGraph

LangGraph allows us to define agents as state graphs. This gives us precise control over the flow of execution.

In [ ]:
import os
os.environ['GOOGLE_API_KEY'] = os.environ.get('GEMINI_API_KEY', 'YOUR_API_KEY')

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from pypdf import PdfReader

# Set up Gemini
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

In [ ]:
@tool
def search_report(query: str) -> str:
    """Search the Apple Environmental Report for information."""
    # A naive exact match search for demonstration
    reader = PdfReader('../data/Apple_Environmental_Progress_Report_2025.pdf')
    text = ''
    for page in reader.pages[:10]: # Read first 10 pages for speed
        text += page.extract_text() + '\n'
    
    # In a real app, use a Vector Store here.
    return text[:2000] # Return a snippet

tools = [search_report]
llm_with_tools = llm.bind_tools(tools)

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

class State(TypedDict):
    messages: Annotated[list, add_messages]

def chatbot(state: State):
    return {'messages': [llm_with_tools.invoke(state['messages'])]}

graph_builder = StateGraph(State)
graph_builder.add_node('chatbot', chatbot)

tool_node = ToolNode(tools=[search_report])
graph_builder.add_node('tools', tool_node)

graph_builder.add_conditional_edges('chatbot', tools_condition)
graph_builder.add_edge('tools', 'chatbot')
graph_builder.add_edge(START, 'chatbot')

graph = graph_builder.compile()

In [ ]:
from langchain_core.messages import HumanMessage

for event in graph.stream({'messages': [HumanMessage(content='What are Apple\'s environmental goals?')] }):
    for value in event.values():
        print('Agent:', value['messages'][-1].content)